# nanoLLaDA: A Toy Implementation of Large Language Diffusion Models

This notebook implements a scaled-down version of **LLaDA** (Large Language Diffusion with mAsking) on the Tiny Shakespeare dataset. It faithfully follows the architecture and diffusion processes described in the paper *Large Language Diffusion Models (Nie et al., 2025)*, adapted for educational purposes.

**Key Features:**
* **Tokenizer:** GPT-2 BPE (via `tiktoken`).
* **Architecture:** Transformer with Bidirectional Attention, RMSNorm, RoPE, and SwiGLU.
* **Training:** Continuous masking rate sampling $t \sim U(0,1)$.
* **Inference:** Iterative "low-confidence re-masking" strategy (Mask-Predict).

## 0. Imports & Configuration

### Library Imports

- **torch** (PyTorch): Core framework for all neural network implementations, tensors, and autograd.
- **rich**: Used for beautiful console logging, status tables, and live training updates.
- **wandb**: Weights & Biases for experiment tracking and visualizing runs.
- **tiktoken**: OpenAI's fast BPE tokenizer for processing text data.
- **numpy / math**: Mathematical operations and array manipulation.
- **matplotlib**: Visualization of training loss and other metrics.
- **tqdm**: Progress bars for data loading and training loops.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import math
import numpy as np

import inspect
from dataclasses import dataclass
from typing import Optional, Tuple
import requests

from tqdm.auto import tqdm
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.live import Live
from rich import box

import matplotlib.pyplot as plt
import tiktoken
import time
import wandb
import os

### Console & Device configuration

In [2]:
# Initialize Rich Console
console = Console()

In [3]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
console.print(f"[bold green]Using device:[/bold green] {device}")
if torch.cuda.is_available():
    console.print(f"[bold blue]GPU:[/bold blue] {torch.cuda.get_device_name(0)}")
    console.print(f"[bold blue]Memory:[/bold blue] {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cpu

### Hyperparameters

In [4]:
batch_size = 12      # How many independent sequences will we process in parallel?
block_size = 256     # What is the maximum context length for predictions?
max_iters = 10000     # Total training steps
eval_interval = 500
learning_rate = 3e-4
device = device
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.1

### WandB Config

In [5]:
use_wandb = False
wandb_project = "nanoLLaDA"
wandb_run_name = "shakespeare-diffusion"
save_model = True
checkpoint_interval = 1000
model_filename = "nanoLLaDA.pth"

## 1. Data Preprocessing

### 1.1 Loading the Dataset
We start by downloading the "Tiny Shakespeare" corpus, a standard dataset used for character-level and token-level language modeling experiments. It consists of roughly 1MB of Shakespeare's writings.

In [6]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text

print(f"Dataset size: {len(text)} characters")
print(f"First 120 characters of the dataset: \n\n{text[:120]}")

Dataset size: 1115394 characters
First 120 characters of the dataset: 

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved ra


### 1.2 Tokenization (BPE) with Custom Mask
We use OpenAI's `tiktoken` (GPT-2 encoding) for Byte Pair Encoding.

**Crucial Step:** We manually add a special `[MASK]` token to the vocabulary.
* Standard GPT-2 does not have a mask token.
* We assign it the ID equal to `enc.n_vocab`.
* We increment the total `vocab_size` by 1 to accommodate it.

We are setting up for a **Masked Language Model (MLM)** task rather than standard causal generation (as shown in Nie et al., 202)

In [7]:
# GPT-2 tokenizer
enc = tiktoken.get_encoding("gpt2")

# We add a special [MASK] token
# We use the vocab_size as our token ID
mask_token_id = enc.n_vocab
vocab_size = enc.n_vocab + 1

print(f"Token {mask_token_id} is [MASK]")
print(f"Vocab Size: {vocab_size}")

Token 50257 is [MASK]
Vocab Size: 50258


### 1.3 Helper Functions
We wrap the tokenizer to handle our custom requirements:
* **`encode`**: Converts string text to integer tokens.
* **`decode`**: Converts integers back to string text. **Note:** This function explicitly filters out the `mask_token_id` before decoding, as the standard `tiktoken` library will crash if it encounters a token ID outside its original vocabulary.

In [8]:
def encode(s):
    return enc.encode(s, allowed_special={"<|endoftext|>"})

In [9]:
def decode(l):
    # Filter out the mask token before decoding, otherwise tiktoken crashes
    clean_ids = [i for i in l if i != mask_token_id]
    return enc.decode(clean_ids)

### 1.4 Train/Validation Split
We tokenize the entire dataset into a single long tensor of integers. We then split this data:
* **90% Training:** Used to update model weights.
* **10% Validation:** Used to evaluate performance on unseen data.

In [10]:
# Split into train/val
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training data tokens {len(train_data)} ")
print(f"Validation data tokens {len(val_data)} ")

Training data tokens 304222 
Validation data tokens 33803 


### 1.5 Data Loader (`get_batch`)
This function generates random batches of inputs and targets for training.
* It selects random starting points (`ix`) in the data.
* It grabs a chunk of data of length `block_size`.

**Note:** Currently, `x` (input) and `y` (target) are clones of each other. The actual **masking logic** (replacing tokens with `[MASK]`) will  occur later inside the training loop or collate function.

In [11]:
# Data Loader
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = x.clone() # The target is the original unmasked sequence
    return x.to(device), y.to(device)

## 2. Model Architecture

LLaDA uses LLaMA-like components: **RMSNorm**, **RoPE** (Rotary Positional Embeddings), and **SwiGLU**. Crucially, it uses **Bidirectional Attention**, meaning there is NO causal mask

### 2.1 Base Components: RMSNorm & RoPE
We implement two critical components standard in modern LLMs (like LLaMA):
1.  **RMSNorm:** A simplified, more stable version of LayerNorm that normalizes input based on root mean square, ignoring the mean.
2.  **RoPE (Rotary Positional Embeddings):** Instead of adding static positional embeddings, we *rotate* the Query and Key vectors.
    * `precompute_freqs_cis`: Pre-calculates complex exponentials for the rotations.
    * `apply_rotary_emb`: Applies the rotation to `q` and `k` tensors.

In [12]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization"""
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

In [13]:
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # complex64
    return freqs_cis

In [14]:
def apply_rotary_emb(xq, xk, freqs_cis):
    # xq.shape = [batch_size, seq_len, n_head, head_dim]
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    
    # Broadcast freq to match batch and head dimensions
    freqs_cis = freqs_cis[:xq.shape[1]].view(1, xq.shape[1], 1, -1)
    
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

### 2.2 Bidirectional Self-Attention
This is the key difference between LLaDA and standard GPT models.
* **No Causal Mask:** We set `is_causal=False`. The model can see the entire sequence (past and future) simultaneously.
* **RoPE Integration:** The rotary embeddings are applied to `xq` and `xk` right before attention is calculated.
* **Efficiency:** We use PyTorch's `F.scaled_dot_product_attention` (SDPA) which automatically uses Flash Attention if available.

In [15]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        
        self.wq = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.wk = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.wv = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.wo = nn.Linear(config.n_embd, config.n_embd, bias=False)
        
        self.dropout = config.dropout
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x, freqs_cis):
        B, T, C = x.shape
        
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)
        
        # Reshape for heads: [B, T, n_head, head_dim]
        xq = xq.view(B, T, self.n_head, self.head_dim)
        xk = xk.view(B, T, self.n_head, self.head_dim)
        xv = xv.view(B, T, self.n_head, self.head_dim)

        # Apply RoPE
        xq, xk = apply_rotary_emb(xq, xk, freqs_cis)

        # Transpose for attention: [B, n_head, T, head_dim]
        xq, xk, xv = xq.transpose(1, 2), xk.transpose(1, 2), xv.transpose(1, 2)

        # Efficient attention (SDPA)
        # Note: is_causal=False for LLaDA (Bidirectional Attention)
        output = F.scaled_dot_product_attention(
            xq, xk, xv, 
            attn_mask=None, 
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False 
        )

        output = output.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.wo(output))

### 2.3 MLP (SwiGLU) & Transformer Block
* **SwiGLU MLP:** We use a Gated Linear Unit with the Swish activation function. It involves three linear projections (`w1`, `w2`, `w3`) instead of the standard two, offering better performance for LLMs.
* **Block:** Combines the Attention and MLP layers using **Pre-Norm** (normalization applied *before* the layer) and residual connections.

In [16]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden_dim = 4 * config.n_embd
        hidden_dim = int(2 * hidden_dim / 3) # SwiGLU convention
        
        self.w1 = nn.Linear(config.n_embd, hidden_dim, bias=False) # Gate
        self.w2 = nn.Linear(config.n_embd, hidden_dim, bias=False) # Value
        self.w3 = nn.Linear(hidden_dim, config.n_embd, bias=False) # Output
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # SwiGLU: (Swish(Gate) * Value) * Output
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))

In [17]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention = CausalSelfAttention(config)
        self.feed_forward = MLP(config)
        self.attention_norm = RMSNorm(config.n_embd)
        self.ffn_norm = RMSNorm(config.n_embd)

    def forward(self, x, freqs_cis):
        h = x + self.attention(self.attention_norm(x), freqs_cis)
        out = h + self.feed_forward(self.ffn_norm(h))
        return out

### 2.4 LLaDA Model & Masked Loss
This class assembles the full model and defines the unique training logic.

* **Architecture:** It ties the weights between the input embedding and the output layer (`token_embedding.weight = output.weight`).
* **Training Objective (Forward Pass):**
    * The model receives `idx` (inputs with `[MASK]` tokens) and `targets` (original original tokens).
    * **Selective Loss:** We calculate CrossEntropy loss **only** on the positions that were masked (`mask_positions = (idx == mask_token_id)`). The model is explicitly trained to "fill in the blanks" rather than predict the next token.

In [18]:
@dataclass
class LLaDAConfig:
    vocab_size: int = vocab_size
    n_layer: int = n_layer
    n_head: int = n_head
    n_embd: int = n_embd
    block_size: int = block_size
    dropout: float = dropout

In [19]:
class LLaDAModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.layers = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.norm = RMSNorm(config.n_embd)
        self.output = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        
        # Tie weights
        self.token_embedding.weight = self.output.weight
        
        # Precompute RoPE frequencies
        self.freqs_cis = precompute_freqs_cis(config.n_embd // config.n_head, config.block_size * 2)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx)
        
        freqs_cis = self.freqs_cis.to(x.device)
        
        for layer in self.layers:
            x = layer(x, freqs_cis)
            
        x = self.norm(x)
        logits = self.output(x)

        loss = None
        if targets is not None:
            # We only calculate loss on the tokens that were MASKED
            # Targets should be the original tokens.
            # idx contains [MASK] (index 0) at masked positions.
            
            # Reshape for CrossEntropy
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), reduction='none')
            
            # Mask optimization: Only learn from masked tokens
            # Create a boolean mask where input is [MASK]
            mask_positions = (idx == mask_token_id).view(-1)
            
            # If no tokens are masked (rare edge case), loss is 0
            if mask_positions.sum() > 0:
                loss = loss[mask_positions].mean()
            else:
                loss = loss.mean() * 0 # No gradient
                
        return logits, loss

## 3. Diffusion Utilities

**Forward Process:**
In LLaDA, at time $t \in [0, 1]$, each token is independently masked with probability $t$.

**Reverse Process (Sampling):**
We start with all tokens masked ($t=1$).
At each step $t$, we predict $x_0$, then sample a new mask for step $t-1$.
The paper uses "Low-Confidence Re-masking": we keep the most confident predictions and re-mask the rest.

### 3.1 Forward Diffusion (Noise Injection)
In continuous diffusion (like Stable Diffusion), forward process adds Gaussian noise. In LLaDA (Discrete Diffusion), **the "noise" is the `[MASK]` token.**

* **Logic:** We sample a time $t \sim U[0, 1]$ and mask that percentage of the tokens.
* **Goal:** The model learns that given a sequence with $t\%$ missing information, it must reconstruct the original $x_0$.

In [20]:
def forward_diffusion(x, mask_token_id=mask_token_id):
    """
    Applies the forward diffusion process (random masking).
    Returns:
        x_masked: The input sequence with some tokens replaced by [MASK]
        mask_ratio: The sampled ratio used
    """
    B, T = x.shape
    # Sample a masking ratio 't' for each sequence in the batch uniformly from [0, 1]
    # LLaDA paper: "masks all tokens randomly at ratio t ~ U[0, 1]"
    t = torch.rand(B, device=x.device) 
    
    # Create a random probability matrix
    probs = torch.rand(B, T, device=x.device)
    
    # Mask where probs < t (broadcast t across T)
    mask = probs < t.view(B, 1)
    
    x_masked = x.clone()
    x_masked[mask] = mask_token_id
    
    return x_masked, t

### 3.2 Reverse Diffusion (Generation)
This implements the **iterative refinement** loop. Unlike GPT (which generates left-to-right), LLaDA generates the entire sequence simultaneously and refines it over `steps`.

**The Algorithm:**
1.  **Predict:** The model predicts the fully unmasked sequence ($x_0$) from the current state.
2.  **Sample:** We select candidate tokens using **Temperature** and **Top-K** sampling (to prevent repetitive, safe loops).
3.  **Re-mask (The Critical Step):**
    * We cannot trust all predictions immediately.
    * We calculate the **confidence** (probability) of every token we just picked.
    * We keep the high-confidence tokens.
    * We **re-mask** the low-confidence tokens so the model can try to predict them again in the next step, conditioned on the "correct" high-confidence tokens.

In [21]:
@torch.no_grad()
def generate(model, prompt_tokens, steps=32, temperature=1.0, top_k=None, device='cuda'):
    """
    Generates text using the Reverse Diffusion process.
    Supports temperature and top-k sampling to avoid repetition.
    """
    model.eval()
    input_ids = torch.tensor(prompt_tokens, dtype=torch.long, device=device).unsqueeze(0)
    unknown_mask = (input_ids == mask_token_id)
    
    for step in range(steps):
        t = 1.0 - (step / steps)
        t_next = 1.0 - ((step + 1) / steps)
        
        logits, _ = model(input_ids)
        
        # --- Sampling Logic ---
        # 1. Apply Temperature
        if temperature > 0:
            logits = logits / temperature
        
        # 2. Apply Top-K Filtering
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, :, -1].unsqueeze(-1)] = -float('Inf')

        probs = F.softmax(logits, dim=-1)
        
        # 3. Sample
        if temperature == 0:
            # Greedy decoding (Argmax) - prone to repetition
            pred_ids = torch.argmax(probs, dim=-1)
            pred_scores = torch.max(probs, dim=-1).values
        else:
            # Stochastic sampling
            # Flatten to sample, then reshape back
            pred_ids = torch.multinomial(probs.view(-1, probs.size(-1)), num_samples=1).view(probs.shape[0], probs.shape[1])
            # Confidence is the probability of the token we actually picked
            pred_scores = torch.gather(probs, -1, pred_ids.unsqueeze(-1)).squeeze(-1)
            
        # Update predictions
        input_ids = torch.where(unknown_mask, pred_ids, input_ids)
        
        # --- Re-masking Strategy ---
        n_masked_next = int(t_next * unknown_mask.sum().item())
        if n_masked_next > 0:
            mask_scores = pred_scores.clone()
            mask_scores[~unknown_mask] = float('inf') 
            _, low_conf_indices = torch.topk(mask_scores.view(-1), k=n_masked_next, largest=False)
            input_ids.view(-1)[low_conf_indices] = mask_token_id
            
    return input_ids[0].tolist()

## 4. Training Loop

### 4.1 Model Initialization
We instantiate the configuration, the model itself, and the optimizer.
* **AdamW:** We use the AdamW optimizer, which is standard for Transformer training.
* **Parameter Count:** We print the total number of trainable parameters to gauge model size (approx 50M for this "nano" version).

In [22]:
model_config = LLaDAConfig()
model = LLaDAModel(model_config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

console.print(f"[bold]Model parameters:[/bold] {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

Model parameters: 29.92M

### 4.2 Weights & Biases Setup
We initialize `wandb` to track our training loss, validation loss, and learning rate over time. This allows for real-time monitoring via the dashboard.

In [23]:
if use_wandb:
    wandb.init(project=wandb_project, name=wandb_run_name, config=model_config.__dict__)

### 4.3 Evaluation Helper (`estimate_loss`)
This function estimates the loss on both split sets (Train/Val) without updating gradients.
* **Why `forward_diffusion` here?** Just like in training, we must **corrupt** the validation data before passing it to the model. The model's task is always "reconstruct the masked tokens."
* **Averaging:** We iterate `eval_iters` times to get a stable average, reducing noise from individual random batches.

In [24]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            X_masked, _ = forward_diffusion(X, mask_token_id)
            _, loss = model(X_masked, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

### 4.4 Main Training Loop
This loop drives the optimization process.
1.  **Data Loading:** Fetch a batch of clean text (`xb`).
2.  **Forward Diffusion (Corruption):** Randomly mask tokens in `xb` based on a random time $t$, creating `xb_masked`.
3.  **Forward Pass:** Feed `xb_masked` into the model. The model tries to predict the original tokens.
4.  **Loss Calculation:** Compare the model's logits against the clean `yb`. (Recall: the loss is only calculated on the masked positions).
5.  **Backpropagation:** Update weights using `optimizer.step()`.
6.  **Logging & Saving:** Periodically check validation loss and save the model if it's the best version seen so far.

In [25]:
# Training
start_time = time.time()
pbar = tqdm(range(max_iters), desc="Training")

best_val_loss = float('inf')

patience = 5
patience_counter = 0
min_delta = 0.001

for iter in pbar:
    xb, yb = get_batch('train')
    xb_masked, _ = forward_diffusion(xb, mask_token_id)
    logits, loss = model(xb_masked, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        val_loss = losses['val']
        
        # Log to WandB
        if use_wandb:
            wandb.log({
                "iter": iter,
                "train/loss": losses['train'],
                "val/loss": losses['val'],
                "lr": learning_rate,
            })

        if val_loss < (best_val_loss - min_delta):
            best_val_loss = val_loss
            patience_counter = 0 # Reset the counter
            
            if use_wandb:
                model_path = os.path.join(wandb.run.dir, "best_model.pth") # Define wandb model path
                torch.save(model.state_dict(), model_path) # Save the model
                wandb.save(model_path, base_path=wandb.run.dir) # Notify wandb
                # Using tqdm.write to avoid breaking progress bar
                tqdm.write(f"New Best model saved! Val loss: {best_val_loss:.4f}")
        else:
            patience_counter += 1 # Incrementa il contatore (nessun miglioramento)
            tqdm.write(f"No improvement. Patience: {patience_counter}/{patience}")
            
        # Update progress bar description
        pbar.set_description(f"Train: {losses['train']:.4f} | Val: {val_loss:.4f} | Pat: {patience_counter}/{patience}")        
        
        # Print nice table to console
        table = Table(title=f"Step {iter}")
        table.add_column("Metric", style="cyan")
        table.add_column("Value", style="magenta")
        table.add_row("Train Loss", f"{losses['train']:.4f}")
        table.add_row("Val Loss", f"{losses['val']:.4f}")
        table.add_row("Patience", f"{patience_counter}/{patience}")
        
        # Use tqdm.write to avoid breaking progress bar
        with console.capture() as capture:
            console.print(table)
        tqdm.write(capture.get())

        if patience_counter >= patience:
            tqdm.write(f"\n Early stopping triggered! Training stopped. Best Val Loss: {best_val_loss:.4f}")
            break

if use_wandb:
    final_path = os.path.join(wandb.run.dir, "final_nanoLLaDA.pth")
    torch.save(model.state_dict(), final_path)
    wandb.save(final_path, base_path=wandb.run.dir)
    wandb.finish()

console.print(f"[bold green]Training finished in {time.time() - start_time:.2f}s[/bold green]")

Training:   0%|          | 0/10000 [00:00<?, ?it/s]


KeyboardInterrupt



## 5. Live Generation Demo (Visualization)

Unlike GPT, which generates one token at a time, LLaDA generates the whole block at once and refines it. Here we visualize this "crystallization" process.

**Setup:**
* **Fixed Context Length:** We must decide the length of the generated sequence in advance (256 tokens) because we initialize the buffer with that many `[MASK]` tokens.
* **Prompting:** We start with "JULIET: " and fill the *rest* of the tensor with `[MASK]`. The model must preserve the prompt and hallucinate the rest.

### Visualizing the Reverse Process
This `Live` loop runs the same `generate` logic as defined earlier, but hooks into `rich.live` to update a table in real-time.

* **Step 0:** The model sees mostly masks. It makes a rough guess of the full sentence.
* **Intermediate Steps:** The code identifies "low confidence" tokens from the previous guess and **re-masks** them.
* **Final Step:** As $t \to 0$, fewer and fewer tokens are re-masked, until the sequence stabilizes into the final output.

*Watch the "Current Sequence" column: you will see the text start as gibberish or vague words and slowly "sharpen" into coherent Shakespearean English.*

In [26]:
# Laoding best model
best_nanoLLaDA_path = "wandb/best_nanoLLaDA/files/best_nanoLLaDA.pth"

if os.path.exists(best_nanoLLaDA_path):
    # map_location assures correct device mapping (cpu/cuda)
    state_dict = torch.load(best_nanoLLaDA_path, map_location=device)
    model.load_state_dict(state_dict)
    console.print(f"[bold green]Best model loaded from: {best_nanoLLaDA_path}[/bold green]")
else:
    console.print(f"[bold red]Warning: {best_nanoLLaDA_path} not found. Using current model state (last epoch).[/bold red]")

model.eval()

Best model loaded from: wandb/best_nanoLLaDA/files/best_nanoLLaDA.pth

LLaDAModel(
  (token_embedding): Embedding(50258, 384)
  (layers): ModuleList(
    (0-5): 6 x Block(
      (attention): CausalSelfAttention(
        (wq): Linear(in_features=384, out_features=384, bias=False)
        (wk): Linear(in_features=384, out_features=384, bias=False)
        (wv): Linear(in_features=384, out_features=384, bias=False)
        (wo): Linear(in_features=384, out_features=384, bias=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (feed_forward): MLP(
        (w1): Linear(in_features=384, out_features=1024, bias=False)
        (w2): Linear(in_features=384, out_features=1024, bias=False)
        (w3): Linear(in_features=1024, out_features=384, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (attention_norm): RMSNorm()
      (ffn_norm): RMSNorm()
    )
  )
  (norm): RMSNorm()
  (output): Linear(in_features=384, out_features=50258, bias=False)
)

In [27]:
# Setup inputs
context_length = 256
prompt_str = "JULIET: "
prompt_ids = encode(prompt_str)
full_input = prompt_ids + [mask_token_id] * (context_length - len(prompt_ids))

curr_ids = torch.tensor(full_input, dtype=torch.long, device=device).unsqueeze(0)
unknown_mask = (curr_ids == mask_token_id)

In [28]:
# Inference params
steps = 64
temperature = 0.8
top_k = 50

In [29]:
console.print(Panel(f"[bold yellow]Generating text with Diffusion (Temp= {temperature} )...[/bold yellow]", title="Demo"))

# Live table for generation steps
table = Table(title="Generation Process", box=box.ROUNDED)
table.add_column("Step", style="dim")
table.add_column("Time (t)", justify="right")
table.add_column("Current Sequence", style="bold white")


with Live(table, console=console, refresh_per_second=4, vertical_overflow="visible") as live:
    for step in range(steps):
        t = 1.0 - (step / steps)
        t_next = 1.0 - ((step + 1) / steps)
        
        logits, _ = model(curr_ids)
        
        # Sampling Logic for Demo
        if temperature > 0:
            logits = logits / temperature
        if top_k is not None:
             v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
             logits[logits < v[:, :, -1].unsqueeze(-1)] = -float('Inf')
        
        probs = F.softmax(logits, dim=-1)
        
        if temperature == 0:
            pred_ids = torch.argmax(probs, dim=-1)
            pred_scores = torch.max(probs, dim=-1).values
        else:
            pred_ids = torch.multinomial(probs.view(-1, probs.size(-1)), num_samples=1).view(probs.shape[0], probs.shape[1])
            pred_scores = torch.gather(probs, -1, pred_ids.unsqueeze(-1)).squeeze(-1)
        
        # Update predictions
        curr_ids = torch.where(unknown_mask, pred_ids, curr_ids)
        
        # Decode for display
        vis_ids = [i for i in curr_ids[0].tolist() if i != mask_token_id]
        decoded_text = enc.decode(vis_ids)
        
        # Add row to table
        table.add_row(f"{step+1}/{steps}", f"{t:.2f}", decoded_text)

        # Force refresh of the live display
        # Uncomment the following line to visualize the generation process
        live.refresh()
        
        # Re-mask
        n_masked_next = int(t_next * unknown_mask.sum().item())
        if n_masked_next > 0:
            mask_scores = pred_scores.clone()
            mask_scores[~unknown_mask] = float('inf')
            _, low_conf_indices = torch.topk(mask_scores.view(-1), k=n_masked_next, largest=False)
            curr_ids.view(-1)[low_conf_indices] = mask_token_id
        
        time.sleep(0.05)

console.print(Panel(decode(curr_ids[0].tolist()), title="Final Output", border_style="green"))

╭───────────────────────────────────────────────────── Demo ──────────────────────────────────────────────────────╮
│ Generating text with Diffusion (Temp= 0.8 )...                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────────── Final Output ──────────────────────────────────────────────────╮
│ JULIET: I will will go.                                                                                         │
│                                                                                                                 │
│ MERGREUTIO:                                                                                                     │
│ As will well marry, I I me all.                                                                                 │
│                                                                                                                 │
│ GREGENCE:                                                                                                       │
│ I not you be gone.                                                                                              │
│                                                                                                                 │
│ STFirstO:                                                                                                       │
│ And do will do?.                                                                                                │
│                                                                                                                 │
│ SecondBoy:                                                                                                      │
│ Hereea, did do ne had that mean hear you thee.                                                                  │
│                                                                                                                 │
│ MERCANIO:                                                                                                       │
│ I he he here.                                                                                                   │
│                                                                                                                 │
│ MERCIIO:                                                                                                        │
│ God I you.                                                                                                      │
│                                                                                                                 │
│ BCIIO:                                                                                                          │
│ Y good Paulman!                                                                                                 │
│                                                                                                                 │
│ BORTENTIO:                                                                                                      │
│ I it I to and I lost do much.                                                                                   │
│                                                                                                                 │
│ MERCIIO:                                                                                                        │
│ Now and if I he no and well you you.                                                                            │
│                                                                                                                 │
│                                                                                                                 │
│  MusENCE:                                                                                                       │
│ I I my you him.                                                                                                 │
│                                                                                                                 │
│ CPageIO:                                              